In [1]:
# files to add to:
files = [
    "n3_traj1_pull_out.npz",
    "n3_traj2_diag_up.npz",
    "n3_traj3_diag_down.npz",
    "n3_traj4_diag2_up.npz",
    "n3_traj5_diag2_down.npz",
    "n3_traj6_vertical_up.npz",
    "n3_traj7_vertical_down.npz",
    "n3_traj8_diagonal_up.npz",
    "n3_traj9_diagonal_down.npz",
    "n3_traj10_diagonal3_up.npz",
    "n3_traj11_diagonal3_down.npz",
    "n3_traj12_diagonal4_up.npz", 
    "n3_traj13_diagonal4_down.npz",
    "n3_traj14_diagonal5_up.npz",
    "n3_traj15_diagonal5_down.npz",
    ]

In [2]:
# add clamped nodes
from pathlib import Path
import sys

import numpy as np


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "examples" / "slinky" / "experiment_data" / "extract_data.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing examples/slinky/experiment_data/extract_data.py")


repo_root = find_repo_root()
experiment_data_dir = repo_root / "examples" / "slinky" / "experiment_data"
if str(experiment_data_dir) not in sys.path:
    sys.path.insert(0, str(experiment_data_dir))

from extract_data import append_right_ghost_clamp, prepend_clamped_node


data_dir = repo_root / "examples" / "slinky" / "simulation_data_2D" / "3_noded"
expected_input_nodes = 3
output_suffix = ""  # set to "_clamped" to write new files instead of replacing the listed files


def n_nodes_from_qs(qs):
    dof = qs.shape[-1]
    if (dof + 1) % 4 != 0:
        raise ValueError(f"qs dof={dof} is not compatible with dof = 4*n_nodes - 1")
    return (dof + 1) // 4


def edge_length_from_first_frame(qs, node_a, node_b):
    a = 4 * node_a
    b = 4 * node_b
    return np.linalg.norm(qs[0, 0, a:a + 3] - qs[0, 0, b:b + 3])


for file_name in files:
    input_path = data_dir / file_name
    output_path = input_path.with_name(f"{input_path.stem}{output_suffix}{input_path.suffix}")

    with np.load(input_path) as data:
        arrays = {key: data[key] for key in data.files}

    qs = arrays["qs"]
    xb = arrays["xb"]
    idx_b = arrays["idx_b"]
    lambdas = arrays["lambdas"]
    valid = arrays.get("valid")

    n_nodes = n_nodes_from_qs(qs)
    if n_nodes != expected_input_nodes:
        raise ValueError(
            f"{file_name} has {n_nodes} nodes; expected {expected_input_nodes}. "
            "This guard prevents accidentally adding clamp nodes twice."
        )

    first_edge_length = edge_length_from_first_frame(qs, 0, 1)
    right_ghost_edge_length = edge_length_from_first_frame(qs, n_nodes - 2, n_nodes - 1)

    qs, xb, idx_b, lambdas, valid = prepend_clamped_node(
        qs=qs,
        xb=xb,
        idx_b=idx_b,
        lambdas=lambdas,
        valid=valid,
        first_edge_length=first_edge_length,
        check_consistency=True,
    )

    qs, xb, idx_b, lambdas, valid = append_right_ghost_clamp(
        qs=qs,
        xb=xb,
        idx_b=idx_b,
        lambdas=lambdas,
        valid=valid,
        ghost_edge_length=right_ghost_edge_length,
        check_consistency=True,
    )

    arrays.update({"qs": qs, "xb": xb, "idx_b": idx_b, "lambdas": lambdas})
    if valid is not None:
        arrays["valid"] = valid

    np.savez(output_path, **arrays)
    print(f"Wrote {output_path.name}: qs {arrays['qs'].shape}, xb {arrays['xb'].shape}, idx_b {arrays['idx_b'].shape}")


Wrote n3_traj1_pull_out.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj2_diag_up.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj3_diag_down.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj4_diag2_up.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj5_diag2_down.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj6_vertical_up.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj7_vertical_down.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj8_diagonal_up.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj9_diagonal_down.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj10_diagonal3_up.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj11_diagonal3_down.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj12_diagonal4_up.npz: qs (1, 150, 19), xb (1, 150, 16), idx_b (16,)
Wrote n3_traj13_diagonal4_down.npz: qs (1, 150, 19), xb (1, 150, 